# Take 1 : Score: 0.50482

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load dataset
path = "/kaggle/input/io-t-sleep-stage-classification-version-2/train/train"
csv_files = glob.glob(f"{path}/*.csv")
df_train = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)

# Feature Scaling
feature_cols = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
scaler = StandardScaler()
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])

# Encode labels
label_encoder = LabelEncoder()
df_train["Sleep_Stage"] = label_encoder.fit_transform(df_train["Sleep_Stage"])

# Prepare X and y
X = df_train[feature_cols].values
y = df_train["Sleep_Stage"].values
num_samples = len(X) // 480
X = X[: num_samples * 480].reshape(num_samples, 480, -1)  # Reshape for LSTM
y = y[::480]

# Use 50% of data
X, _, y, _ = train_test_split(X, y, test_size=0.5, random_state=42)

# Convert labels to categorical
y_categorical = tf.keras.utils.to_categorical(y, num_classes=3)

# Split dataset
X_train, X_val, y_train, y_val = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

# Define LSTM Model
model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(480, X.shape[2])),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(64),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train LSTM Model
model.fit(X_train, y_train, epochs=20, batch_size=64, validation_data=(X_val, y_val), shuffle=True)

# Load test data
test_data_path = "/kaggle/input/io-t-sleep-stage-classification-version-2/test_segment/test_segment"
test_folders = ["test001", "test002", "test003", "test004", "test005", "test006", "test007", "test008", "test009", "test010"]
test_csv_files = []

for folder in test_folders:
    folder_path = os.path.join(test_data_path, folder)
    test_csv_files.extend(glob.glob(f"{folder_path}/*.csv"))

# Load and preprocess test data
df_test = pd.concat([pd.read_csv(file) for file in test_csv_files], ignore_index=True)
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

# Reshape test data into sequences
num_samples_test = len(df_test) // 480
X_test = df_test[feature_cols].values[: num_samples_test * 480].reshape(num_samples_test, 480, -1)

# Predict sleep stages
y_test_pred = np.argmax(model.predict(X_test), axis=1)
y_test_pred_labels = label_encoder.inverse_transform(y_test_pred)

# Load sample submission
submission_path = "/kaggle/input/io-t-sleep-stage-classification-version-2/sample_submission.csv"
df_submission = pd.read_csv(submission_path)

# Ensure y_test_pred_labels matches df_submission length
df_submission = df_submission.iloc[:len(y_test_pred_labels)].copy()
df_submission["labels"] = y_test_pred_labels

# Keep only id and labels columns
df_submission = df_submission[["id", "labels"]]

# Save final submission
final_submission_path = "/kaggle/working/submission.csv"
df_submission.to_csv(final_submission_path, index=False)
print("Submission file saved at:", final_submission_path)

# Take 2 : Scores 0.50219

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load dataset
path = "/kaggle/input/io-t-sleep-stage-classification-version-2/train/train"
csv_files = glob.glob(f"{path}/*.csv")
df_train = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)

# Feature Scaling
feature_cols = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
scaler = StandardScaler()
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])

# Encode labels
label_encoder = LabelEncoder()
df_train["Sleep_Stage"] = label_encoder.fit_transform(df_train["Sleep_Stage"])

# Prepare X and y
X = df_train[feature_cols].values
y = df_train["Sleep_Stage"].values
num_samples = len(X) // 480  # Adjust window size (Try 480, 600, 360)
X = X[: num_samples * 480].reshape(num_samples, 480, -1)  # Reshape for LSTM
y = y[::480]

# Use 50% of data
X, _, y, _ = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert labels to categorical
y_categorical = tf.keras.utils.to_categorical(y, num_classes=3)

# Split dataset
X_train, X_val, y_train, y_val = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

# Define LSTM Model
model = Sequential([
    Bidirectional(LSTM(128, return_sequences=True, input_shape=(480, X.shape[2]))),
    BatchNormalization(),
    Dropout(0.3),
    Bidirectional(LSTM(128)),
    Dropout(0.3),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(3, activation='softmax')
])

# Compile Model
model.compile(
    optimizer=Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
early_stopping = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

# Train LSTM Model
model.fit(
    X_train, y_train,
    epochs=30,  # Increased epochs
    batch_size=128,  # Adjusted batch size
    validation_data=(X_val, y_val),
    shuffle=True,
    callbacks=[lr_scheduler, early_stopping]
)

# Load test data
test_data_path = "/kaggle/input/io-t-sleep-stage-classification-version-2/test_segment/test_segment"
test_folders = ["test001", "test002", "test003", "test004", "test005", "test006", "test007", "test008", "test009", "test010"]
test_csv_files = []

for folder in test_folders:
    folder_path = os.path.join(test_data_path, folder)
    test_csv_files.extend(glob.glob(f"{folder_path}/*.csv"))

# Load and preprocess test data
df_test = pd.concat([pd.read_csv(file) for file in test_csv_files], ignore_index=True)
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

# Reshape test data into sequences
num_samples_test = len(df_test) // 480
X_test = df_test[feature_cols].values[: num_samples_test * 480].reshape(num_samples_test, 480, -1)

# Predict sleep stages
y_test_pred = np.argmax(model.predict(X_test), axis=1)
y_test_pred_labels = label_encoder.inverse_transform(y_test_pred)

# Load sample submission
submission_path = "/kaggle/input/io-t-sleep-stage-classification-version-2/sample_submission.csv"
df_submission = pd.read_csv(submission_path)

# Ensure y_test_pred_labels matches df_submission length
df_submission = df_submission.iloc[:len(y_test_pred_labels)].copy()
df_submission["labels"] = y_test_pred_labels

# Keep only id and labels columns
df_submission = df_submission[["id", "labels"]]

# Save final submission
final_submission_path = "/kaggle/working/submission.csv"
df_submission.to_csv(final_submission_path, index=False)
print("Submission file saved at:", final_submission_path)

# Take 3 : Scores 0.40816

In [ ]:
import pandas as pd
import numpy as np
import glob
import os
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers.schedules import ExponentialDecay
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Load dataset
path = "/kaggle/input/io-t-sleep-stage-classification-version-2/train/train"
csv_files = glob.glob(f"{path}/*.csv")
df_train = pd.concat([pd.read_csv(file) for file in csv_files], ignore_index=True)

# Feature Scaling
feature_cols = ["BVP", "ACC_X", "ACC_Y", "ACC_Z", "TEMP", "EDA", "HR", "IBI"]
scaler = StandardScaler()
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])

# Encode labels
label_encoder = LabelEncoder()
df_train["Sleep_Stage"] = label_encoder.fit_transform(df_train["Sleep_Stage"])

# Prepare X and y
X = df_train[feature_cols].values
y = df_train["Sleep_Stage"].values
num_samples = len(X) // 480  # Adjust window size to 480
X = X[: num_samples * 480].reshape(num_samples, 480, -1)  # Reshape for LSTM
y = y[: num_samples * 480].reshape(num_samples, 480)[:, 0]  # Ensure y matches X

# Debugging: Check shapes
print(f"X shape: {X.shape}, y shape: {y.shape}")

# Handle class imbalance using SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X.reshape(X.shape[0], -1), y)
X_resampled = X_resampled.reshape(-1, 480, X.shape[2])  # Convert back to 3D

# Use 100% of data
X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

# Convert labels to categorical
y_categorical = tf.keras.utils.to_categorical(y_train, num_classes=3)
y_val_categorical = tf.keras.utils.to_categorical(y_val, num_classes=3)

# Define Learning Rate Schedule
lr_schedule = ExponentialDecay(
    initial_learning_rate=0.0001, decay_steps=10000, decay_rate=0.9, staircase=True
)

# Define LSTM Model
model = Sequential([
    Bidirectional(LSTM(256, return_sequences=True, input_shape=(480, X_train.shape[2]))),
    BatchNormalization(),
    Dropout(0.5),
    Bidirectional(LSTM(256)),
    Dropout(0.5),
    Dense(512, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

# Compile Model
optimizer = Adam(learning_rate=lr_schedule)  # Adjusted Learning Rate
model.compile(
    optimizer=optimizer,
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Callbacks
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)
early_stopping = EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True)

# Train LSTM Model
model.fit(
    X_train, y_categorical,
    epochs=20,  # Adjusted epochs
    batch_size=128,  # Adjusted batch size
    validation_data=(X_val, y_val_categorical),
    shuffle=True,
    callbacks=[lr_scheduler, early_stopping]
)

# Load test data
test_data_path = "/kaggle/input/io-t-sleep-stage-classification-version-2/test_segment/test_segment"
test_folders = ["test001", "test002", "test003", "test004", "test005", "test006", "test007", "test008", "test009", "test010"]
test_csv_files = []

for folder in test_folders:
    folder_path = os.path.join(test_data_path, folder)
    test_csv_files.extend(glob.glob(f"{folder_path}/*.csv"))

# Load and preprocess test data
df_test = pd.concat([pd.read_csv(file) for file in test_csv_files], ignore_index=True)
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

# Reshape test data into sequences
num_samples_test = len(df_test) // 480
X_test = df_test[feature_cols].values[: num_samples_test * 480].reshape(num_samples_test, 480, -1)

# Predict sleep stages
y_test_pred = np.argmax(model.predict(X_test), axis=1)
y_test_pred_labels = label_encoder.inverse_transform(y_test_pred)

# Load sample submission
submission_path = "/kaggle/input/io-t-sleep-stage-classification-version-2/sample_submission.csv"
df_submission = pd.read_csv(submission_path)

# Ensure y_test_pred_labels matches df_submission length
df_submission = df_submission.iloc[:len(y_test_pred_labels)].copy()
df_submission["labels"] = y_test_pred_labels

# Keep only id and labels columns
df_submission = df_submission[["id", "labels"]]

# Save final submission
final_submission_path = "/kaggle/working/submission2.csv"
df_submission.to_csv(final_submission_path, index=False)
print("Submission file saved at:", final_submission_path)